# Stage 1 & 2A: Data Cleaning, Missingness Analysis & Univariate EDA
### AMEO 2015 Engineering Placement Benchmark (2,998 Records, 34 Features)
**Assigned to:** Dhruven  
**Course:** IT462 Exploratory Data Analysis | Syllabus Modules 1, 2, 4, 5, 6, 7  

---
## 📌 Syllabus Modules Covered in this Notebook
1. **Module 1: Data and Statistics** (Data Science Pipeline, Role of EDA in ML, Structured vs Unstructured, Population vs Sample)
2. **Module 2: Measurement Scales** (Nominal, Ordinal, Interval, Ratio, Discrete vs Continuous Variables)
3. **Module 4: Data Import & Cleaning** (Inspection, Insights, Duplicate Detection, Inconsistent Formats, Invalid Entries, Type Conversion)
4. **Module 5: Missing Data Analysis** (MCAR, MAR, MNAR Mechanisms, Missingness Visualization, Missing Data Treatment)
5. **Module 6: Descriptive Summary** (Central Tendency, Dispersion, MAD, Coefficient of Variation, Skewness, Kurtosis, Tukey's Five-Number Summary)
6. **Module 7: Univariate Analysis** (Histograms, Density Plots / KDE, Boxplots, Violin Plots, Bar Charts)


## 1. Setup & Environment Configuration
Import essential numerical and statistical libraries (`numpy`, `pandas`, `scipy.stats`, `matplotlib`, `seaborn`, `missingno`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Optional missingness visualization library
try:
    import missingno as msno
except ImportError:
    msno = None

# Styling setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Environment configured successfully.")
if msno:
    print("missingno library loaded.")
else:
    print("missingno not installed; fallback to seaborn missingness heatmaps.")

## 2. Foundations: Data and Statistics (Syllabus Module 1)
### Statistical Definitions & Taxonomy
- **What is Data?** Facts, figures, or information collected together for reference or analysis. In our context, data comprises academic records, cognitive psychometrics, and employment outcomes for engineering graduates.
- **The Data Science Pipeline:**
  1. *Collection:* Sourcing real empirical records (Aspiring Minds AMCAT study).
  2. *Data Cleaning:* Rectifying anomalies, sentinel codes (`-1`), missing values, and formatting discrepancies.
  3. *Exploratory Data Analysis (EDA):* Summarizing distributional shapes, detecting outliers, computing correlation, and discovering patterns.
  4. *Feature Engineering & Transformation:* Correcting skewness, encoding categorical tiers, standardizing scales.
  5. *Predictive Modeling:* Supervised learning benchmarks.
  6. *Data Storytelling & Dashboards:* Communicating actionable insights to university leadership.
- **Role of EDA in Machine Learning:** Proposed by John Tukey (1977), EDA serves as the foundation before formal modeling to understand data characteristics, check statistical assumptions (normality, multicollinearity, homoscedasticity), formulate hypotheses, and prevent "garbage in, garbage out".
- **Structured vs Unstructured Data:** Our AMEO dataset is purely **Structured Tabular Data** ($N = 2,998$ records $\times$ 34 well-defined attribute columns with rigid relational schema), contrasted with unstructured data (e.g., student resumes as PDFs, interview audio, coding video recordings).
- **Population vs Sample:**
  - *Target Population:* All ~1.5 million engineering graduates graduating annually across all Indian AICTE institutions.
  - *Empirical Sample:* $n = 2,998$ engineering graduates who took the standardized AMCAT assessment.

## 3. Data Import & Initial Inspection (Syllabus Module 4)
> **TODO (Dhruven):** Load `../data/raw/Engineering_graduate_salary.csv`, inspect `.shape`, `.info()`, `.head()`, and examine data types.

In [ ]:
raw_path = '../data/raw/Engineering_graduate_salary.csv'
df = pd.read_csv(raw_path)
print(f"Loaded Raw Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.info()
df.head()

## 4. Measurement Scales & Variable Types (Syllabus Module 2)
Categorizing the 34 attributes according to Stevens' (1946) Four Measurement Scales and Discrete vs Continuous nature:

| Scale | Mathematical Properties | AMEO Dataset Features | Permissible Statistics |
| :--- | :--- | :--- | :--- |
| **Nominal** | Identity, equality ($=, \neq$) | `Gender`, `10board`, `12board`, `Degree`, `Specialization`, `CollegeState`, `JobCity` | Mode, Frequency, Contingency Tables, Chi-Square |
| **Ordinal** | Order, rank ($>, <$) | `CollegeTier` (1 vs 2), `CollegeCityTier` (0 vs 1) | Median, Percentiles, Rank Correlation (Spearman) |
| **Interval** | Meaningful differences ($+, -$), arbitrary zero | `GraduationYear`, `12graduation`, `conscientiousness`, `agreeableness`, `extraversion`, `nueroticism`, `openess_to_experience` (Standardized z-scores) | Mean, Standard Deviation, Pearson Correlation |
| **Ratio** | True absolute zero, meaningful ratios ($\times, \div$) | `Salary` (INR), `collegeGPA`, `10percentage`, `12percentage`, `English`, `Logical`, `Quant`, `Domain`, `ComputerProgramming` | Geometric Mean, Coefficient of Variation (CV) |

### Discrete vs Continuous Variables:
- **Discrete Variables:** Finite or countably infinite values (e.g., `GraduationYear`, `CollegeTier`, `CollegeCityTier`).
- **Continuous Variables:** Uncountably infinite values within an interval (e.g., `Salary`, `collegeGPA`, `10percentage`, `Quant`, `agreeableness`).

In [ ]:
# TODO (Dhruven):
# Categorize the 34 attributes into Stevens' Measurement Scales and Discrete vs Continuous.
# Define scale dictionaries and inspect feature data types.
# Example:
# scales = {'Nominal': [...], 'Ordinal': [...], 'Interval': [...], 'Ratio': [...]}
# print(scales)


## 5. Data Cleaning, Validation & Datetime Parsing (Syllabus Module 4 & Lecture Notes)
### Pipeline Sequence from Lecture Notes (Slide 1):
1. **Step 1: Import Data** (`pd.read_csv`)
2. **Step 2: Validation / Prescreening** (Shape, null detection, invalid codes)
3. **Step 2b: Time Series / Datetime Parsing & Timedelta:**
   - Parse date string features: `df['DOB_parsed'] = pd.to_datetime(df['DOB'])`
   - Compute timedelta features: $\text{Age at Graduation} = \text{GraduationYear} - \text{DOB Year}$
4. **Step 3: Cleaning & Pre-processing:**
   - Duplicate detection (`df.duplicated().sum()`)
   - Invalid & Sentinel codes: Handle `-1` unattempted optional technical module scores (`CivilEngg`, `MechanicalEngg`, `ComputerProgramming`).

> **TODO (Dhruven):** Implement duplicate checks, datetime parsing on `DOB`, and sentinel `-1` handling.

In [ ]:
# TODO (Dhruven):
# 1. Step 2: Check for duplicate rows using df.duplicated().sum()
# 2. Step 2b: Parse 'DOB' to datetime (pd.to_datetime) and compute student age/timedelta at graduation
# 3. Step 3: Inspect sentinel value (-1) in optional technical domain scores (ComputerProgramming, CivilEngg, etc.)
#    and decide treatment (imputation vs indicator flag)


## 6. Missing Data Analysis: MCAR, MAR, MNAR (Syllabus Module 5)
### Missingness Mechanisms:
1. **MCAR (Missing Completely at Random):** Missingness is entirely independent of observed and unobserved data (e.g., random network packet drop during test upload).
2. **MAR (Missing at Random):** Missingness depends systematically on observed attributes but not on the missing value itself (e.g., Mechanical Engineering students did not take the `ComputerScience` module because of their `Specialization`).
3. **MNAR (Missing Not at Random):** Missingness depends directly on the unobserved value itself (e.g., students who performed poorly in an optional test chose not to submit their score).

> **TODO (Dhruven):** Analyze null counts, visualize missingness matrix or heatmap, and state the missingness mechanism.

In [ ]:
# TODO (Dhruven):
# 1. Check df.isnull().sum() for all columns
# 2. Visualize missingness patterns using missingno (matrix/heatmap) or seaborn heatmap
# 3. Classify missingness mechanisms as MCAR, MAR, or MNAR with justification in markdown


## 7. Descriptive Summaries: Shape, Dispersion & Central Tendency (Syllabus Module 6)
Compute rigorous statistical descriptors across all continuous variables:
- **Measures of Central Tendency:** Mean, Median, Mode
- **Measures of Dispersion / Variation:** Variance ($\sigma^2$), Standard Deviation ($\sigma$), Interquartile Range ($IQR = Q_3 - Q_1$)
- **Median Absolute Deviation (MAD):** $\text{MAD} = \text{median}(|X_i - \tilde{X}|)$ — Tukey's robust outlier-resistant dispersion metric
- **Coefficient of Variation (CV):** $CV = \frac{\sigma}{\mu} \times 100\%$ — scale-independent relative dispersion
- **Measures of Shape:** Fisher-Pearson Skewness (asymmetry) & Kurtosis (tailedness/heaviness of tails)
- **Tukey's Five-Number Summary:** Minimum, First Quartile ($Q_1$), Median, Third Quartile ($Q_3$), Maximum

> **TODO (Dhruven):** Build a consolidated descriptive statistics table.

In [ ]:
# TODO (Dhruven):
# Compute consolidated descriptive statistical summary table for continuous numeric features:
# 1. Central Tendency: Mean, Median, Mode
# 2. Dispersion: Variance, Standard Deviation, IQR, Range
# 3. Robust / Relative metrics: MAD (scipy.stats.median_abs_deviation), CV (Std/Mean * 100)
# 4. Shape: Fisher-Pearson Skewness (stats.skew), Kurtosis (stats.kurtosis)
# 5. Tukey's Five-Number Summary: Min, Q1, Median, Q3, Max
# Consolidate into a summary DataFrame (e.g. summary_df)


## 8. Univariate Visualizations (Syllabus Module 7)
Visualizing the individual distributions using the 5 core univariate techniques specified in the syllabus:
1. **Histogram:** Frequency distribution of `Salary` (identifying right skewness).
2. **Density Plot (KDE):** Continuous empirical density estimation of `collegeGPA` and `Quant`.
3. **Boxplot:** Tukey boxplot displaying median, whiskers, and extreme outliers.
4. **Violin Plot:** Combining KDE density shape with boxplot quartiles.
5. **Bar Chart:** Frequency breakdown of categorical attributes (`Gender`, `CollegeTier`, `Degree`).

> **TODO (Dhruven):** Plot univariate figures and save key plots to `../reports/figures/`.

In [ ]:
# TODO (Dhruven):
# Generate univariate visualizations and save key plots to ../reports/figures/:
# 1. Histogram & KDE: Frequency distribution of Salary (inspect right skewness)
# 2. Tukey Boxplot: collegeGPA (identify outliers)
# 3. Violin Plot: Quant or Logical score (density shape + quartiles)
# 4. Bar Chart: Categorical frequency (CollegeTier, Gender, Degree)


## 9. Export Cleaned Dataset for Stage 2B (Prit)
> **TODO (Dhruven):** Once cleaning and invalid code handling is finalized, export the clean dataset to `../data/processed/ameo_cleaned.csv`.

In [ ]:
# TODO (Dhruven): Uncomment to export after completing Stage 1 & 2A cleaning
# os.makedirs('../data/processed', exist_ok=True)
# df.to_csv('../data/processed/ameo_cleaned.csv', index=False)
# print("Cleaned dataset saved successfully to ../data/processed/ameo_cleaned.csv")